# Optimization From Scratch: SGD, Momentum, RMSProp, Adam

Companion notebook for the [Optimization](https://codemia.ai/courses/deep_learning_foundations/optimization) lesson on Codemia.

**What you will build**: four optimizers (SGD, Momentum, RMSProp, Adam) implemented from scratch in NumPy, trained on a toy 2D classification problem, with their paths visualized on a contour plot of the loss landscape.

**Karpathy companion**: [Building makemore Part 3: Activations, Gradients, BatchNorm](https://www.youtube.com/watch?v=P6sfmUTpUmc) walks through optimizer intuition in the same hands-on style.

*Adapted in part from karpathy/makemore (MIT) — the manual-gradient style is borrowed from that repo.*

**Time**: ~45 minutes  
**GPU required**: no (pure NumPy on CPU)

## Setup

In [ ]:
# Only numpy + matplotlib. We are deliberately *not* using torch here — writing
# the update rules by hand forces you to internalize what each optimizer does.
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

# Fixed seed so the contour plot, data, and init are reproducible across runs.
np.random.seed(0)

## Concept recap

All four optimizers follow the same shape: pick a direction from the gradient, scale it, and step. The differences are:

- **SGD**: step in the raw gradient direction. Simple, but zig-zags in ravines.
- **Momentum**: accumulate an exponentially-decaying average of past gradients. The inertia smooths out zig-zags.
- **RMSProp**: keep a running average of the *squared* gradient and divide the step by its square root. Per-parameter learning rates; tames directions where gradients are huge.
- **Adam**: combine Momentum (first moment) with RMSProp (second moment). Bias-correct both.

We will implement all four, then watch them crawl across the same loss landscape.

## Implementation

### Step 1 — A toy 2D classification dataset

Two interleaving half-circles. Simple enough to debug by eye; non-linear enough that a tiny MLP actually has to learn.

In [ ]:
def make_moons(n=200, noise=0.2):
    # Half-moon dataset — classic non-linearly separable toy.
    # We roll our own instead of using sklearn to keep dependencies minimal.
    n_per_class = n // 2
    theta = np.linspace(0, np.pi, n_per_class)
    # Upper moon, class 0
    x0 = np.stack([np.cos(theta), np.sin(theta)], axis=1)
    # Lower moon, class 1 — shifted so they interleave
    x1 = np.stack([1 - np.cos(theta), 0.5 - np.sin(theta)], axis=1)
    X = np.concatenate([x0, x1], axis=0) + noise * np.random.randn(n, 2)
    y = np.concatenate([np.zeros(n_per_class), np.ones(n_per_class)]).astype(np.int64)
    return X.astype(np.float32), y

X, y = make_moons(n=400, noise=0.15)
print('X shape:', X.shape, 'y shape:', y.shape, 'class balance:', y.mean())

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=15)
plt.title('Half-moons'); plt.axis('equal'); plt.show()

### Step 2 — A tiny MLP with hand-written forward and backward

Two hidden layers, tanh activations, softmax+cross-entropy at the output. We keep the parameter count tiny so the loss surface we will plot later is actually interpretable.

In [ ]:
def init_params(in_dim=2, hidden=8, out_dim=2, scale=0.5):
    # Kaiming-ish init scaled for tanh. Small net so we can visualize trajectories.
    return {
        'W1': np.random.randn(in_dim, hidden).astype(np.float32) * scale,
        'b1': np.zeros(hidden, dtype=np.float32),
        'W2': np.random.randn(hidden, out_dim).astype(np.float32) * scale,
        'b2': np.zeros(out_dim, dtype=np.float32),
    }

def forward(params, X):
    # z1 pre-activation, h1 post-activation. We cache everything needed for backward.
    z1 = X @ params['W1'] + params['b1']
    h1 = np.tanh(z1)
    logits = h1 @ params['W2'] + params['b2']
    return logits, (X, z1, h1)

def softmax_ce(logits, y):
    # Numerically stable softmax + cross-entropy. Shift by max so exp() does not overflow.
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    probs = exp / exp.sum(axis=1, keepdims=True)
    nll = -np.log(probs[np.arange(len(y)), y] + 1e-12)
    return nll.mean(), probs

In [ ]:
def backward(params, cache, probs, y):
    # Manual backprop through the two-layer MLP. Each grad derived from first principles.
    X, z1, h1 = cache
    N = X.shape[0]
    # dL/dlogits for softmax+CE is (probs - one_hot)/N — the famously clean result.
    dlogits = probs.copy()
    dlogits[np.arange(N), y] -= 1
    dlogits /= N
    # Second layer: logits = h1 @ W2 + b2
    dW2 = h1.T @ dlogits
    db2 = dlogits.sum(axis=0)
    dh1 = dlogits @ params['W2'].T
    # tanh derivative: 1 - tanh(z)^2
    dz1 = dh1 * (1 - h1 * h1)
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0)
    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

### Step 3 — The four optimizers

Each optimizer keeps its own state dict. The API is identical so we can drop any of them into the same training loop.

In [ ]:
class SGD:
    # Vanilla SGD. No memory, no adaptation. The baseline every other optimizer compares against.
    def __init__(self, params, lr=0.1):
        self.lr = lr
    def step(self, params, grads):
        for k in params:
            params[k] -= self.lr * grads[k]

class Momentum:
    # Exponentially-decayed velocity. Imagine a ball rolling downhill: gradient is force, v is speed.
    def __init__(self, params, lr=0.1, beta=0.9):
        self.lr, self.beta = lr, beta
        # Velocity has the same shape as the parameter it tracks.
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
    def step(self, params, grads):
        for k in params:
            # v <- beta*v + (1-beta)*g. We keep (1-beta) so lr is comparable across optimizers.
            self.v[k] = self.beta * self.v[k] + (1 - self.beta) * grads[k]
            params[k] -= self.lr * self.v[k]

In [ ]:
class RMSProp:
    # Running average of g^2. Dividing by sqrt(E[g^2]) gives per-parameter adaptive lr.
    def __init__(self, params, lr=0.05, beta=0.9, eps=1e-8):
        self.lr, self.beta, self.eps = lr, beta, eps
        self.s = {k: np.zeros_like(v) for k, v in params.items()}
    def step(self, params, grads):
        for k in params:
            self.s[k] = self.beta * self.s[k] + (1 - self.beta) * grads[k] ** 2
            params[k] -= self.lr * grads[k] / (np.sqrt(self.s[k]) + self.eps)

class Adam:
    # Momentum (m) + RMSProp (v) + bias correction. The default optimizer for a reason.
    def __init__(self, params, lr=0.05, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0  # timestep drives the bias correction
    def step(self, params, grads):
        self.t += 1
        for k in params:
            self.m[k] = self.beta1 * self.m[k] + (1 - self.beta1) * grads[k]
            self.v[k] = self.beta2 * self.v[k] + (1 - self.beta2) * grads[k] ** 2
            # Bias-correct: early in training m and v are tiny because we init at 0.
            m_hat = self.m[k] / (1 - self.beta1 ** self.t)
            v_hat = self.v[k] / (1 - self.beta2 ** self.t)
            params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

### Step 4 — Training loop

We track two things per step: the loss and a snapshot of two chosen parameters so we can plot their 2D path through the loss landscape.

In [ ]:
def train(opt_cls, steps=400, **opt_kwargs):
    # Fresh init each run so every optimizer starts from the same point. This is what makes
    # the trajectory comparison meaningful: only the update rule differs.
    np.random.seed(1)
    params = init_params()
    opt = opt_cls(params, **opt_kwargs)
    losses = []
    # Track two scalar params that move: one weight in the first layer and one in the second.
    traj = []
    for step in range(steps):
        logits, cache = forward(params, X)
        loss, probs = softmax_ce(logits, y)
        grads = backward(params, cache, probs, y)
        opt.step(params, grads)
        losses.append(loss)
        traj.append((float(params['W1'][0, 0]), float(params['W2'][0, 0])))
        if step % 100 == 0:
            # Print loss and a representative weight so you can see learning happen.
            print(f'  step {step:4d}  loss={loss:.4f}  W1[0,0]={params["W1"][0,0]:+.4f}')
    return losses, np.array(traj), params

## Experiment

In [ ]:
results = {}
for name, (cls, kw) in {
    'SGD':      (SGD,      {'lr': 0.3}),
    'Momentum': (Momentum, {'lr': 0.3, 'beta': 0.9}),
    'RMSProp':  (RMSProp,  {'lr': 0.05}),
    'Adam':     (Adam,     {'lr': 0.1}),
}.items():
    print(f'\n=== Training with {name} ===')
    losses, traj, params = train(cls, steps=400, **kw)
    results[name] = (losses, traj, params)
    print(f'  final loss: {losses[-1]:.4f}')

In [ ]:
# Convergence curves — log scale because SGD and Adam live in very different neighborhoods.
plt.figure(figsize=(7, 4))
for name, (losses, _, _) in results.items():
    plt.plot(losses, label=name)
plt.yscale('log')
plt.xlabel('step'); plt.ylabel('loss (log)')
plt.title('Convergence speed on half-moons'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

### Visualizing the loss landscape

We slice the loss along two parameter axes: `W1[0,0]` and `W2[0,0]`. Every other parameter is frozen to whatever Adam converged to. The resulting 2D contour is a faithful cross-section of the full loss surface near the optimum, and the trajectories show how each optimizer navigates it.

In [ ]:
# Use Adam's final params as the frozen backdrop.
_, _, base = results['Adam']
base = {k: v.copy() for k, v in base.items()}

def loss_at(w1, w2):
    p = {k: v.copy() for k, v in base.items()}
    p['W1'][0, 0] = w1
    p['W2'][0, 0] = w2
    logits, _ = forward(p, X)
    loss, _ = softmax_ce(logits, y)
    return loss

# Compute over a grid around the initial weight so all trajectories fit in frame.
grid = np.linspace(-2.5, 2.5, 60)
W1g, W2g = np.meshgrid(grid, grid)
Z = np.vectorize(loss_at)(W1g, W2g)
print('loss grid range:', Z.min(), '..', Z.max())

In [ ]:
plt.figure(figsize=(7, 6))
cs = plt.contourf(W1g, W2g, Z, levels=30, cmap=cm.viridis)
plt.colorbar(cs, label='loss')
colors = {'SGD': 'white', 'Momentum': 'orange', 'RMSProp': 'red', 'Adam': 'magenta'}
for name, (_, traj, _) in results.items():
    plt.plot(traj[:, 0], traj[:, 1], '-', color=colors[name], lw=2, label=name)
    plt.plot(traj[0, 0], traj[0, 1], 'o', color=colors[name])
    plt.plot(traj[-1, 0], traj[-1, 1], '*', color=colors[name], ms=14)
plt.xlabel('W1[0,0]'); plt.ylabel('W2[0,0]')
plt.title('Optimizer trajectories on a 2D slice of the loss surface')
plt.legend(); plt.show()

## Variations

Two small ablations to build intuition:

1. Crank Adam's learning rate up 5x and watch it overshoot.
2. Turn off Adam's bias correction and see how much it slows down early training.

In [ ]:
# 1. Adam with a bad learning rate
print('Adam lr=0.5 (too hot):')
losses_hot, _, _ = train(Adam, steps=400, lr=0.5)
print('Adam lr=0.1 (baseline):')
losses_ok, _, _ = train(Adam, steps=400, lr=0.1)

plt.figure(figsize=(6, 3))
plt.plot(losses_hot, label='lr=0.5 (overshoots)')
plt.plot(losses_ok, label='lr=0.1 (baseline)')
plt.yscale('log'); plt.legend(); plt.title('Adam sensitivity to lr'); plt.show()

In [ ]:
# 2. Adam without bias correction.
class AdamNoBias(Adam):
    def step(self, params, grads):
        self.t += 1
        for k in params:
            self.m[k] = self.beta1 * self.m[k] + (1 - self.beta1) * grads[k]
            self.v[k] = self.beta2 * self.v[k] + (1 - self.beta2) * grads[k] ** 2
            params[k] -= self.lr * self.m[k] / (np.sqrt(self.v[k]) + self.eps)

print('Adam no bias correction:')
losses_nb, _, _ = train(AdamNoBias, steps=400, lr=0.1)

plt.figure(figsize=(6, 3))
plt.plot(losses_ok, label='Adam (with bias correction)')
plt.plot(losses_nb, label='Adam (no bias correction)')
plt.yscale('log'); plt.legend(); plt.title('Bias correction matters in the first ~100 steps'); plt.show()

## What to try next

- Add **AdamW**: decouple weight decay from the gradient update and compare regularization strength.
- Plot the *effective* learning rate per parameter (`lr / (sqrt(v) + eps)`) for RMSProp — verify that high-gradient directions shrink.
- Run the same four optimizers on a **narrow ravine** loss (e.g. `f(x, y) = 100 * x**2 + y**2`) and watch SGD zig-zag while Momentum flies down.
- Swap the toy problem for MNIST (still 2 layers) and see whether the relative ordering holds.